<a href="https://colab.research.google.com/github/hulwene/DoAnML/blob/main/DenseNet_Workspace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 DoAnML - DenseNet-40 Custom
**Kiến trúc**: DenseNet-40 (L=40, k=12) | **Nâng cấp**: Mish + SE Block | **Dataset**: CIFAR-10

### Cấu trúc dự án
| File | Chức năng |
|------|----------|
| `model.py` | Kiến trúc DenseNet (SEBlock, DenseLayer, DenseBlock, TransitionLayer, DenseNetCustom) |
| `data_loader.py` | Tải và xử lý dữ liệu CIFAR-10 (Data Augmentation) |
| `train.py` | Hàm huấn luyện, đánh giá, lưu checkpoint |
| `utils.py` | Vẽ biểu đồ Loss/Accuracy |

## 1. Cài đặt Môi trường

In [ ]:
from google.colab import drive
import os

# Kết nối với Google Drive
drive.mount('/content/drive')

# Tạo thư mục dự án nếu chưa có và di chuyển vào đó
project_folder = '/content/drive/MyDrive/DenseNet_Project'
if not os.path.exists(project_folder):
    os.makedirs(project_folder)

%cd {project_folder}

In [ ]:
import os

# 1. Tên repo
REPO_NAME = "DoAnML"

# 2. Clone repo từ GitHub (nếu chưa có)
if not os.path.exists(REPO_NAME):
    !git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git

# 3. Di chuyển vào thư mục DoAnML
%cd {REPO_NAME}

# 4. Cài đặt thư viện
!pip install -r requirements.txt

# 5. Kiểm tra GPU
import torch
print(f"GPU đang sử dụng: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "Chưa bật GPU!")

## 2. Tải Dữ liệu CIFAR-10

In [ ]:
from data_loader import get_data_loaders

trainloader, testloader = get_data_loaders()

print(f"Số batch huấn luyện: {len(trainloader)}")
print(f"Số batch kiểm thử: {len(testloader)}")

## 3. Khởi tạo Mô hình DenseNet-40 Custom

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from model import DenseNetCustom

# 1. Thiết bị tính toán
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Số vòng huấn luyện
num_epochs = 50

# 3. Khởi tạo mô hình DenseNet-40 (k=12) tự xây dựng
#    - growth_rate=12: Mỗi lớp xuất ra đúng 12 bản đồ đặc trưng mới
#    - block_config=(12,12,12): 3 Dense Block × 12 lớp → L = 12×3 + 4 = 40
#    - Đã tích hợp: Mish activation + SE Block
model = DenseNetCustom(
    growth_rate=12,
    block_config=(12, 12, 12),
    num_classes=10
)
model = model.to(device)

# In tổng số tham số
total_params = sum(p.numel() for p in model.parameters())
print(f"Tổng số tham số: {total_params:,}")

# 4. Hàm mất mát và Bộ tối ưu
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 5. Cosine Annealing Scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

print(f"✅ Đã khởi tạo mô hình DenseNet-40 custom trên: {device}")

## 4. Huấn luyện Mô hình

In [ ]:
from train import train_one_epoch, evaluate, save_checkpoint

# Mảng lưu lịch sử để vẽ biểu đồ
train_losses, train_accuracies = [], []
test_losses, test_accuracies = [], []

print("🚀 BẮT ĐẦU HUẤN LUYỆN...")
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 20)

    # Chạy Train và Test
    train_loss, train_acc = train_one_epoch(model, trainloader, optimizer, criterion, device)
    test_loss, test_acc = evaluate(model, testloader, criterion, device)

    # Cập nhật tốc độ học
    scheduler.step()

    # Lưu lại kết quả
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    test_losses.append(test_loss)
    test_accuracies.append(test_acc)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Test  Loss: {test_loss:.4f}  | Test  Acc: {test_acc:.2f}%")
    print(f"Learning Rate hiện tại: {scheduler.get_last_lr()[0]:.6f}")

    # Lưu model mỗi 10 epoch
    if (epoch + 1) % 10 == 0:
        save_checkpoint(model, optimizer, scheduler, epoch + 1)

print("\n🎉 HOÀN TẤT HUẤN LUYỆN!")

## 5. Trực quan hóa Kết quả

In [ ]:
from utils import plot_training_history

plot_training_history(train_losses, test_losses, train_accuracies, test_accuracies)